# 👕 Lookzi — Virtual Try-On (Colab)

**Muhim:** Runtime → Change runtime type → **T4 GPU** tanlang!

Qadamlar:
1. Hamma celllarni ketma-ket ishga tushiring (Ctrl+F9)
2. 4-cell models yuklab oladi (~27 GB, 15-30 daqiqa)
3. Oxirgi cellda Gradio linki chiqadi — uni bosing

In [ ]:
# ── 1. GPU tekshirish ─────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# ── 2. Kutubxonalar o'rnatish ─────────────────────────────────────────────────
import subprocess, sys

packages = [
    'diffusers==0.27.2',
    'transformers==4.46.3',
    'accelerate==1.1.1',
    'huggingface_hub==0.26.5',
    'gradio==4.44.0',
    'onnxruntime',
    'einops',
    'opencv-python-headless',
    'scipy',
]

for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

print('\n✅ All packages installed!')

In [ ]:
# ── 3. IDM-VTON kodni yuklab olish ───────────────────────────────────────────
import os

if not os.path.exists('/content/IDM-VTON'):
    os.system('git clone https://github.com/yisol/IDM-VTON.git /content/IDM-VTON')
    print('✅ Repo cloned')
else:
    print('✅ Repo already exists')

# src/ fayllarini TemryL versiyasidan olamiz (diffusers 0.27.2 bilan mos)
if not os.path.exists('/content/IDM-VTON-src'):
    os.system('git clone https://github.com/TemryL/ComfyUI-IDM-VTON.git /content/IDM-VTON-src')
    print('✅ Patched src cloned')
else:
    print('✅ Patched src already exists')

os.listdir('/content/IDM-VTON')

In [ ]:
# ── 4. Modellarni yuklab olish (~27 GB, birinchi marta 15-30 daqiqa) ─────────
# Keyingi safar Drive ga saqlash uchun Google Drive mount qiling:
# from google.colab import drive; drive.mount('/content/drive')
# Va MODEL_DIR ni '/content/drive/MyDrive/IDM-VTON-models' ga o'zgartiring

from huggingface_hub import snapshot_download
import os

MODEL_DIR = '/content/IDM-VTON-models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('Downloading IDM-VTON models (~27 GB)...')
print('Bu 15-30 daqiqa ketadi...')

snapshot_download(
    repo_id='yisol/IDM-VTON',
    local_dir=MODEL_DIR,
    ignore_patterns=['*.md', '*.txt', '.gitattributes'],
)

print('\n✅ Models downloaded!')
print('Contents:', os.listdir(MODEL_DIR))

In [ ]:
# ── 5. Path setup va import fix ───────────────────────────────────────────────
import sys, os

IDM_VTON    = '/content/IDM-VTON'
IDM_SRC     = '/content/IDM-VTON-src'
MODEL_DIR   = '/content/IDM-VTON-models'
PREPROCESS  = os.path.join(IDM_VTON, 'gradio_demo', 'preprocess')

# src/ — TemryL versiyasi (diffusers 0.27.2 bilan mos, relative import tuzatilgan)
SRC_DIR = os.path.join(IDM_SRC, 'src', 'idm_vton')

for p in [
    IDM_VTON,
    IDM_VTON + '/gradio_demo',
    SRC_DIR,
    PREPROCESS,
    PREPROCESS + '/openpose',
    PREPROCESS + '/openpose/annotator',
]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

# ip_adapter — TemryL repo dan
IP_ADAPTER = os.path.join(IDM_SRC, 'src', 'idm_vton')
if IP_ADAPTER not in sys.path:
    sys.path.insert(0, IP_ADAPTER)

# OpenPose weights path
try:
    import annotator.util as au
    au.annotator_ckpts_path = os.path.join(MODEL_DIR, 'openpose', 'ckpts')
    print('✅ OpenPose path set')
except Exception as e:
    print(f'OpenPose path warning: {e}')

# unet_hacked_tryon.py dagi relative import ni absolute ga o'zgartirish
tryon_file = os.path.join(SRC_DIR, 'unet_hacked_tryon.py')
if os.path.exists(tryon_file):
    with open(tryon_file, 'r') as f:
        content = f.read()
    content = content.replace(
        'from ..ip_adapter.ip_adapter import Resampler',
        'from ip_adapter.ip_adapter import Resampler'
    ).replace(
        'from ..ip_adapter.attention_processor import IPAttnProcessor2_0 as IPAttnProcessor, AttnProcessor2_0 as AttnProcessor',
        'from ip_adapter.attention_processor import IPAttnProcessor2_0 as IPAttnProcessor, AttnProcessor2_0 as AttnProcessor'
    )
    with open(tryon_file, 'w') as f:
        f.write(content)
    print('✅ unet_hacked_tryon.py patched')

print('✅ Paths configured')

In [ ]:
# ── 6. Modellarni yuklash ─────────────────────────────────────────────────────
import os, torch, warnings
warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import numpy as np
import onnxruntime as ort
from PIL import Image
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image

from transformers import (
    CLIPImageProcessor, CLIPVisionModelWithProjection,
    CLIPTextModel, CLIPTextModelWithProjection, AutoTokenizer,
)
from diffusers import DDPMScheduler, AutoencoderKL

from unet_hacked_tryon   import UNet2DConditionModel
from unet_hacked_garmnet import UNet2DConditionModel as UNet2DConditionModel_ref
from tryon_pipeline      import StableDiffusionXLInpaintPipeline as TryonPipeline

from preprocess.openpose.run_openpose import OpenPose
from preprocess.humanparsing.parsing_api import onnx_inference

# utils_mask — gradio_demo dan
sys.path.insert(0, '/content/IDM-VTON/gradio_demo')
from utils_mask import get_mask_location

DEVICE = 'cuda'
DTYPE  = torch.float16
W, H   = 768, 1024
MODELS = '/content/IDM-VTON-models'

print('Loading models (2-3 minutes)...')

# Human Parsing
class Parsing:
    def __init__(self):
        opts = ort.SessionOptions()
        self.session     = ort.InferenceSession(os.path.join(MODELS, 'humanparsing', 'parsing_atr.onnx'),
                                                sess_options=opts, providers=['CPUExecutionProvider'])
        self.lip_session = ort.InferenceSession(os.path.join(MODELS, 'humanparsing', 'parsing_lip.onnx'),
                                                sess_options=opts, providers=['CPUExecutionProvider'])
    def __call__(self, img):
        return onnx_inference(self.session, self.lip_session, img)

parsing_model  = Parsing()
openpose_model = OpenPose(0)
print('  ✅ OpenPose + Parsing ready')

unet = UNet2DConditionModel.from_pretrained(
    MODELS, subfolder='unet', torch_dtype=DTYPE).requires_grad_(False).eval()
unet_encoder = UNet2DConditionModel_ref.from_pretrained(
    MODELS, subfolder='unet_encoder', torch_dtype=DTYPE).requires_grad_(False).eval()
vae = AutoencoderKL.from_pretrained(
    MODELS, subfolder='vae', torch_dtype=DTYPE).requires_grad_(False).eval()
text_enc1 = CLIPTextModel.from_pretrained(
    MODELS, subfolder='text_encoder', torch_dtype=DTYPE).requires_grad_(False).eval()
text_enc2 = CLIPTextModelWithProjection.from_pretrained(
    MODELS, subfolder='text_encoder_2', torch_dtype=DTYPE).requires_grad_(False).eval()
img_enc = CLIPVisionModelWithProjection.from_pretrained(
    MODELS, subfolder='image_encoder', torch_dtype=DTYPE).requires_grad_(False).eval()
tok1  = AutoTokenizer.from_pretrained(MODELS, subfolder='tokenizer',   use_fast=False)
tok2  = AutoTokenizer.from_pretrained(MODELS, subfolder='tokenizer_2', use_fast=False)
sched = DDPMScheduler.from_pretrained(MODELS, subfolder='scheduler')
print('  ✅ UNet, VAE, encoders ready')

pipe = TryonPipeline.from_pretrained(
    MODELS, unet=unet, vae=vae,
    feature_extractor=CLIPImageProcessor(),
    text_encoder=text_enc1, text_encoder_2=text_enc2,
    tokenizer=tok1, tokenizer_2=tok2,
    scheduler=sched, image_encoder=img_enc,
    torch_dtype=DTYPE,
)
pipe.unet_encoder = unet_encoder

# T4/A100 da hamma modellar GPU ga sig'adi — GPU-swap kerak emas
pipe.to(DEVICE)
pipe.unet_encoder.to(DEVICE)
pipe.enable_attention_slicing(1)
pipe.enable_vae_slicing()

tensor_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

print('\n✅ All models ready!')
print(f'VRAM used: {torch.cuda.memory_allocated()/1024**3:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
# ── 7. Gradio app ishga tushirish ─────────────────────────────────────────────
import gradio as gr
import gc

def run_tryon(person_np, garment_np, garment_desc,
              auto_mask, steps, seed, progress=gr.Progress()):
    if person_np is None or garment_np is None:
        return None, None, 'Upload both a person photo and a garment photo.'

    try:
        person  = Image.fromarray(person_np).convert('RGB').resize((W, H))
        garment = Image.fromarray(garment_np).convert('RGB').resize((W, H))

        # Mask
        if auto_mask:
            progress(0.10, 'OpenPose keypoint detection...')
            keypoints = openpose_model(person.resize((384, 512)))
            progress(0.20, 'Human parsing...')
            parse_result, _ = parsing_model(person.resize((384, 512)))
            progress(0.30, 'Building clothing mask...')
            mask, _ = get_mask_location('hd', 'upper_body', parse_result, keypoints)
            mask = mask.resize((W, H))
        else:
            from PIL import ImageDraw
            mask = Image.new('L', (W, H), 0)
            ImageDraw.Draw(mask).rectangle(
                [int(W*0.05), int(H*0.10), int(W*0.95), int(H*0.72)], fill=255)

        mask_gray_t  = (1 - tensor_tf(mask)) * tensor_tf(person)
        mask_preview = to_pil_image(((mask_gray_t + 1.0) / 2.0).clamp(0, 1))

        # Encode prompts
        progress(0.40, 'Encoding prompts...')
        neg = 'monochrome, lowres, bad anatomy, worst quality, low quality'
        with torch.no_grad():
            p_emb, n_emb, p_pool, n_pool = pipe.encode_prompt(
                f'model is wearing {garment_desc}',
                num_images_per_prompt=1,
                do_classifier_free_guidance=True,
                negative_prompt=neg,
            )
            c_emb, _, _, _ = pipe.encode_prompt(
                [f'a photo of {garment_desc}'],
                num_images_per_prompt=1,
                do_classifier_free_guidance=False,
                negative_prompt=[''],
            )

        # Diffusion
        progress(0.50, f'Running diffusion ({int(steps)} steps)...')
        pose_t    = tensor_tf(person).unsqueeze(0).to(DEVICE, DTYPE)  # placeholder
        garment_t = tensor_tf(garment).unsqueeze(0).to(DEVICE, DTYPE)
        gen = torch.Generator(DEVICE).manual_seed(int(seed))

        with torch.no_grad(), torch.cuda.amp.autocast(), torch.inference_mode():
            images = pipe(
                prompt_embeds=p_emb.to(DEVICE, DTYPE),
                negative_prompt_embeds=n_emb.to(DEVICE, DTYPE),
                pooled_prompt_embeds=p_pool.to(DEVICE, DTYPE),
                negative_pooled_prompt_embeds=n_pool.to(DEVICE, DTYPE),
                num_inference_steps=int(steps),
                generator=gen,
                strength=1.0,
                pose_img=garment_t,      # Colab: DensePose o'rniga garment ishlatamiz
                text_embeds_cloth=c_emb.to(DEVICE, DTYPE),
                cloth=garment_t,
                mask_image=mask,
                image=person,
                height=H, width=W,
                ip_adapter_image=garment,
                guidance_scale=2.0,
            )[0]

        torch.cuda.empty_cache()
        return images[0], mask_preview, 'Done!'

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None, None, 'GPU out of memory — try fewer steps'
    except Exception as e:
        import traceback
        return None, None, f'Error: {e}\n{traceback.format_exc()}'


with gr.Blocks(title='Lookzi — Virtual Try-On', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 👕 Lookzi — Virtual Try-On\nUpload a person photo and a garment, then click **Try It On**.')

    with gr.Row():
        with gr.Column():
            person_in  = gr.Image(label='Person Photo',  type='numpy', height=420)
        with gr.Column():
            garment_in = gr.Image(label='Garment Photo', type='numpy', height=420)
        with gr.Column():
            result_out = gr.Image(label='Result',                  height=420)
            mask_out   = gr.Image(label='Detected clothing area',  height=200)
            status_out = gr.Textbox(label='Status', interactive=False, lines=1)

    desc_in      = gr.Textbox(label='Garment description',
                              placeholder='e.g. white cotton t-shirt',
                              value='a shirt')
    auto_mask_cb = gr.Checkbox(label='Auto-detect clothing area (recommended)', value=True)

    with gr.Accordion('Advanced settings', open=False):
        steps_sl = gr.Slider(15, 40, value=30, step=1, label='Denoising steps')
        seed_nb  = gr.Number(value=42, label='Seed', precision=0)

    run_btn = gr.Button('✨ Try It On', variant='primary', size='lg')
    run_btn.click(
        fn=run_tryon,
        inputs=[person_in, garment_in, desc_in, auto_mask_cb, steps_sl, seed_nb],
        outputs=[result_out, mask_out, status_out],
    )

# share=True — public URL beradi (72 soat)
demo.launch(share=True, debug=True)